In [1]:
import pandas as pd
import numpy as np
import random
import json
from datetime import datetime, timedelta

In [7]:
# Load TAK repository to get valid concepts
with open('../intervene_ar/config/tak-repo-portable.json', 'r', encoding='utf-8') as f:
    tak_data = json.load(f)

# Extract only derived concepts (not raw concepts)
all_concepts = tak_data['taks']
derived_concepts = {name: obj for name, obj in all_concepts.items() 
                   if obj.get('derived_from') is not None}

# Categorize concepts by type
# Events and Patterns: duration = 1s
events_patterns = [name for name in derived_concepts.keys() 
                   if name.endswith('_EVENT') or name.endswith('_PATTERN')]

# States, Trends, Contexts: duration > 1s
states_trends_contexts = [name for name in derived_concepts.keys() 
                          if name.endswith('_STATE') or name.endswith('_TREND') or name.endswith('_CONTEXT')]

print(f"Found {len(derived_concepts)} derived concepts:")
print(f"  - {len(events_patterns)} Events/Patterns (1s duration)")
print(f"  - {len(states_trends_contexts)} States/Trends/Contexts (longer duration)")

# Configurable parameters
NUM_PATIENTS = 500
EVENTS_RANGE = (50, 500)
HOSPITALIZATION_DAYS_RANGE = (3, 14)
START_DATE = datetime(2023, 1, 1)
TOTAL_DAYS = 14
REPO = 'train' # 'train' / 'test'

# Outcomes - manually specified by user
OUTCOMES = [
    "DISGLYCEMIA_EVENT",
    "KIDNEY_COMPLICATION_EVENT",
    "ACUTE_RESPIRATORY_DISORDER",
    "INFECTION_EVENT",
    "ATHEROSCLEROSIS_EVENT",
    "CARDIO-VASCULAR_DISORDER_EVENT",
    "NERVOUS_SYSTEM_DISORDER_EVENT",
    "NEUROVASCULAR_COMPLICATION_EVENT",
    "SKIN_ULCER_EVENT",
    "OTHER_COMPLICATION_EVENT",
    "RETINOPATHY_EVENT",
    "HYPEROSMOLALITY_EVENT",
    "DIABETIC_COMA_EVENT",
    "KETOACIDOSIS_EVENT",
    "ACIDOSIS_EVENT",
]

Found 108 derived concepts:
  - 39 Events/Patterns (1s duration)
  - 60 States/Trends/Contexts (longer duration)


In [8]:
def generate_patient_data(patient_id):
    stay_length_days = random.randint(*HOSPITALIZATION_DAYS_RANGE)
    patient_start = START_DATE + timedelta(days=random.randint(0, TOTAL_DAYS - stay_length_days))
    stay_seconds = stay_length_days * 86400

    rows = []

    # Admission
    admission_time = patient_start
    rows.append({
        'PatientId': f'{patient_id:04d}',
        'ConceptName': 'ADMISSION_EVENT',
        'StartDateTime': admission_time,
        'EndDateTime': admission_time + timedelta(seconds=1),
        'Value': 'True'
    })

    # Meals per day
    for day in range(stay_length_days):
        base_date = admission_time + timedelta(days=day)
        for meal, hour in zip(["Breakfast", "Lunch", "Dinner"], [7, 12, 17]):
            ts = base_date.replace(hour=hour, minute=0, second=0)
            rows.append({
                'PatientId': f'{patient_id:04d}',
                'ConceptName': 'MEAL_CONTEXT',
                'StartDateTime': ts,
                'EndDateTime': ts + timedelta(seconds=1),
                'Value': meal
            })

    # Random medical events using valid concepts from TAK repo
    n_events = random.randint(*EVENTS_RANGE)
    for _ in range(n_events):
        ts = admission_time + timedelta(seconds=random.randint(0, stay_seconds - 1))
        
        # 60% states/trends/contexts (longer duration), 40% events/patterns (1s duration)
        if random.random() < 0.6 and states_trends_contexts:
            concept = random.choice(states_trends_contexts)
            # For states/trends, append a value suffix
            if '_STATE' in concept:
                suffix = random.choice(['High', 'Medium', 'Low'])
            elif '_TREND' in concept:
                suffix = random.choice(['INC', 'DEC', 'Stable'])
            elif '_CONTEXT' in concept:
                suffix = random.choice(['True', 'False'])
            else:
                suffix = 'True'
            duration = timedelta(seconds=random.randint(360, 7200))  # between 6 min and 2 hours
        else:
            concept = random.choice(events_patterns) if events_patterns else random.choice(states_trends_contexts)
            suffix = 'True'
            duration = timedelta(seconds=1)

        rows.append({
            'PatientId': f'{patient_id:04d}',
            'ConceptName': concept,
            'StartDateTime': ts,
            'EndDateTime': ts + duration,
            'Value': suffix
        })

    # Complications
    complications = []
    if random.random() < 0.55:
        complications = random.sample([o for o in OUTCOMES if o not in ['DEATH_EVENT', 'RELEASE_EVENT']],
                                      k=random.randint(1, 3))
    terminal = 'DEATH_EVENT' if random.random() < 0.2 else 'RELEASE_EVENT'
    outcome_sequence = complications + [terminal]

    outcome_time = patient_start + timedelta(seconds=stay_seconds + 1)
    for i, outcome in enumerate(outcome_sequence):
        rows.append({
            'PatientId': f'{patient_id:04d}',
            'ConceptName': outcome,
            'StartDateTime': outcome_time + timedelta(seconds=i + 1),
            'EndDateTime': outcome_time + timedelta(seconds=i + 2),
            'Value': 'True'
        })

    return rows

# Generate all patient data
all_rows = []
for pid in range(NUM_PATIENTS):
    all_rows.extend(generate_patient_data(pid))

df = pd.DataFrame(all_rows)

# Sort by PatientId and StartDateTime
df = df.sort_values(by=['PatientId', 'StartDateTime']).reset_index(drop=True)

# Format datetime columns for CSV
df['StartDateTime'] = df['StartDateTime'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['EndDateTime'] = df['EndDateTime'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Display summary
print(f"\nGenerated {len(df)} events for {NUM_PATIENTS} patients")
print(f"Unique concepts used: {df['ConceptName'].nunique()}")
print(f"\nTop 10 concepts:")
print(df['ConceptName'].value_counts().head(10))

# Save temporal data
df.to_csv(f"{REPO}/synthetic_diabetes_temporal_data.csv", index=False)
print(f"\nSaved temporal data to {REPO}/synthetic_diabetes_temporal_data.csv")

# Generate context data
context_data = pd.DataFrame({
    'PatientId': sorted(df['PatientId'].unique()),
    'Age': np.random.randint(18, 86, size=len(df['PatientId'].unique())),
    'Gender': np.random.randint(0, 2, size=len(df['PatientId'].unique()))  # 0 or 1
})

# Save context data to CSV
context_data.to_csv(f"{REPO}/synthetic_diabetes_context_data.csv", index=False)
print(f"Saved context data to {REPO}/synthetic_diabetes_context_data.csv")


Generated 147658 events for 500 patients
Unique concepts used: 100

Top 10 concepts:
ConceptName
MEAL_CONTEXT                                   13794
ADMISSION_EVENT                                 1938
RELEASE_EVENT                                   1763
OTHER_COMPLICATION_EVENT                        1481
DEATH_EVENT                                     1468
RETINOPATHY_EVENT                               1458
CONTINUE_METFORMIN_ON_ADMISSION_PATTERN         1438
HEPARIN_BITZUA_EVENT                            1434
ROUTINE_GLUCOSE_MEASURE_ON_STEROIDS_PATTERN     1432
KIDNEY_COMPLICATION_EVENT                       1430
Name: count, dtype: int64

Saved temporal data to train/synthetic_diabetes_temporal_data.csv
Saved context data to train/synthetic_diabetes_context_data.csv


In [4]:
# Display sample of generated data
print("Sample of generated temporal data:")
df.head(50)

Sample of generated temporal data:


,PatientId,ConceptName,StartDateTime,EndDateTime,Value
0,0000,ADMISSION_EVENT,2023-01-01 00:00:00,2023-01-01 00:00:01,True
1,0000,MEAL_CONTEXT,2023-01-01 07:00:00,2023-01-01 07:00:01,Breakfast
2,0000,MEAL_CONTEXT,2023-01-01 12:00:00,2023-01-01 12:00:01,Lunch
3,0000,PLT_MEASURE_STATE,2023-01-01 12:29:28,2023-01-01 12:55:08,High
4,0000,INFECTION_WBC_MEASURE_TREND,2023-01-01 12:43:32,2023-01-01 13:12:58,INC
5,0000,GLUCOSE_MEASURE_TREND,2023-01-01 13:51:48,2023-01-01 15:01:35,DEC
6,0000,CALCIUM-GLUCONATE_BITZUA_EVENT,2023-01-01 16:47:53,2023-01-01 16:47:54,True
7,0000,MEAL_CONTEXT,2023-01-01 17:00:00,2023-01-01 17:00:01,Dinner
8,0000,TROPONIN_MEASURE_STATE,2023-01-01 17:41:55,2023-01-01 18:23:58,High
9,0000,STOP_ANTIDIABETICS_ON_ADMISSION_PATTERN,2023-01-01 17:55:32,2023-01-01 17:55:33,True


In [ ]:
# Verify all concepts are valid (derived concepts only)
temporal_concepts = set(df['ConceptName'].unique())
raw_concepts = {name for name, obj in all_concepts.items() 
                if obj.get('derived_from') is None and name not in ['ADMISSION_EVENT', 'MEAL_CONTEXT'] + OUTCOMES}

invalid_concepts = temporal_concepts.intersection(raw_concepts)
if invalid_concepts:
    print(f"⚠️  WARNING: Found {len(invalid_concepts)} raw concepts in generated data:")
    print(invalid_concepts)
else:
    print("✅ All generated concepts are valid (derived or special concepts)")